In [101]:
!pip install -q streamlit pyngrok


In [102]:
%%writefile app.py

import streamlit as st
import pandas as pd

st.set_page_config(
    page_title="Student Progress Tracker",
    page_icon="🎓",
    layout="wide"
)

st.title("🎓 Student Progress Tracking & Performance Prediction")

st.write(
    "Track student performance over four months "
    "and predict semester performance."
)

st.divider()

# Student Information
st.header("👨‍🎓 Student Information")

col1, col2 = st.columns(2)

with col1:
    student_name = st.text_input("Student Name")
    student_id = st.text_input("Student ID")
    course = st.text_input("Course")

with col2:
    department = st.text_input("Department")

    semester = st.selectbox(
        "Semester",
        [
            "1st",
            "2nd",
            "3rd",
            "4th",
            "5th",
            "6th",
            "7th",
            "8th"
        ]
    )

    attendance = st.number_input(
        "Overall Attendance (%)",
        min_value=0.0,
        max_value=100.0,
        value=75.0
    )

study_hours = st.number_input(
    "Average Study Hours per Day",
    min_value=0.0,
    max_value=24.0,
    value=2.0
)

st.divider()

# Subject Information
st.header("📚 Subject Performance")

num_subjects = st.number_input(
    "Number of Subjects",
    min_value=1,
    max_value=12,
    value=4,
    step=1
)

subjects = []

for i in range(int(num_subjects)):

    st.subheader(f"Subject {i + 1}")

    subject_name = st.text_input(
        "Subject Name",
        key=f"subject_name_{i}",
        placeholder="Example: Mathematics"
    )

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        month_1 = st.number_input(
            "Month 1",
            min_value=0.0,
            max_value=100.0,
            key=f"month1_{i}"
        )

    with col2:
        month_2 = st.number_input(
            "Month 2",
            min_value=0.0,
            max_value=100.0,
            key=f"month2_{i}"
        )

    with col3:
        month_3 = st.number_input(
            "Month 3",
            min_value=0.0,
            max_value=100.0,
            key=f"month3_{i}"
        )

    with col4:
        month_4 = st.number_input(
            "Month 4",
            min_value=0.0,
            max_value=100.0,
            key=f"month4_{i}"
        )

    subjects.append({
        "Subject": subject_name,
        "Month 1": month_1,
        "Month 2": month_2,
        "Month 3": month_3,
        "Month 4": month_4
    })

st.divider()

if st.button("🔍 Analyze Student", type="primary"):

    if not student_name:
        st.error("Please enter student name.")

    elif not student_id:
        st.error("Please enter student ID.")

    elif any(subject["Subject"].strip() == "" for subject in subjects):
        st.error("Please enter all subject names.")

    else:

        df = pd.DataFrame(subjects)

        # Average performance
        df["Average"] = df[
            ["Month 1", "Month 2", "Month 3", "Month 4"]
        ].mean(axis=1)

        # Performance trend
        def calculate_trend(row):

            first_half = (
                row["Month 1"] +
                row["Month 2"]
            ) / 2

            second_half = (
                row["Month 3"] +
                row["Month 4"]
            ) / 2

            difference = second_half - first_half

            if difference >= 5:
                return "Improving"

            elif difference <= -5:
                return "Declining"

            else:
                return "Stable"

        df["Trend"] = df.apply(
            calculate_trend,
            axis=1
        )

        overall_average = df["Average"].mean()

        st.success("Analysis generated successfully!")

        # Overview
        st.header("📊 Student Overview")

        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.metric(
                "Overall Average",
                f"{overall_average:.2f}%"
            )

        with col2:
            st.metric(
                "Attendance",
                f"{attendance:.1f}%"
            )

        with col3:
            st.metric(
                "Study Hours",
                f"{study_hours:.1f}"
            )

        with col4:
            improving = (
                df["Trend"] == "Improving"
            ).sum()

            st.metric(
                "Improving Subjects",
                improving
            )

        # Subject table
        st.subheader("📚 Subject-wise Performance")

        st.dataframe(
            df,
            use_container_width=True,
            hide_index=True
        )

        # Chart
        st.subheader("📈 Performance Trend")

        chart_data = df.set_index("Subject")[
            [
                "Month 1",
                "Month 2",
                "Month 3",
                "Month 4"
            ]
        ]

        st.line_chart(chart_data)

        # Category
        st.subheader("🎯 Current Performance")

        if overall_average >= 80:
            st.success("Excellent Performance")

        elif overall_average >= 60:
            st.info("Good Performance")

        elif overall_average >= 40:
            st.warning("Average Performance")

        else:
            st.error("At Risk")

        # Recommendations
        st.subheader("💡 Recommendations")

        for _, row in df.iterrows():

            if row["Trend"] == "Declining":

                st.warning(
                    f"⚠️ {row['Subject']}: "
                    "performance is declining. "
                    "Consider additional revision."
                )

            elif row["Average"] < 50:

                st.warning(
                    f"⚠️ {row['Subject']}: "
                    "current performance is low. "
                    "More practice is recommended."
                )

            elif row["Trend"] == "Improving":

                st.success(
                    f"✅ {row['Subject']}: "
                    "performance is improving. "
                    "Keep up the current progress."
                )

Overwriting app.py


In [103]:
import os

DATA_DIR = "/content/oulad"

os.makedirs(DATA_DIR, exist_ok=True)

print("Dataset folder created:", DATA_DIR)

Dataset folder created: /content/oulad


In [104]:
!streamlit run app.py &> /content/streamlit.log &

In [105]:
from pyngrok import ngrok

ngrok.set_auth_token("")

public_url = ngrok.connect(8501)

print("Open this URL:")
print(public_url)

Open this URL:
NgrokTunnel: "https://freeness-outbid-balcony.ngrok-free.dev" -> "http://localhost:8501"


In [106]:
import pandas as pd

DATA_DIR = "/content/oulad"

student_info = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")
student_assessment = pd.read_csv(f"{DATA_DIR}/studentAssessment.csv")
assessments = pd.read_csv(f"{DATA_DIR}/assessments.csv")
student_vle = pd.read_csv(f"{DATA_DIR}/studentVle.csv")
student_registration = pd.read_csv(f"{DATA_DIR}/studentRegistration.csv")
courses = pd.read_csv(f"{DATA_DIR}/courses.csv")

print("All files loaded.")

All files loaded.


In [107]:
print("student_info")
display(student_info.head())

print("student_assessment")
display(student_assessment.head())

print("assessments")
display(assessments.head())

student_info


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


student_assessment


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79


assessments


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10.0
1,AAA,2013J,1753,TMA,54,20.0
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
4,AAA,2013J,1756,TMA,215,30.0


In [108]:
print("Shapes:")
print("student_info:", student_info.shape)
print("student_assessment:", student_assessment.shape)
print("assessments:", assessments.shape)
print("student_vle:", student_vle.shape)
print("student_registration:", student_registration.shape)
print("courses:", courses.shape)

Shapes:
student_info: (32593, 12)
student_assessment: (173912, 5)
assessments: (206, 6)
student_vle: (10655280, 6)
student_registration: (32593, 5)
courses: (22, 3)


In [109]:
print("Missing values:\n")

print("student_info")
print(student_info.isnull().sum())

print("\nstudent_assessment")
print(student_assessment.isnull().sum())

print("\nassessments")
print(assessments.isnull().sum())

print("\nstudent_vle")
print(student_vle.isnull().sum())

print("\nstudent_registration")
print(student_registration.isnull().sum())

print("\ncourses")
print(courses.isnull().sum())

Missing values:

student_info
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

student_assessment
id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64

assessments
code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

student_vle
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

student_registration
code_module            0
code_presentation      0
id_student             0
date_registration      0
date_unregistration    0
dtype: int64

courses
code_module    

In [110]:
assessment_data = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

display(assessment_data.head(10))

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight
0,1752,11391,18,0,78,AAA,2013J,TMA,19,10.0
1,1752,28400,22,0,70,AAA,2013J,TMA,19,10.0
2,1752,31604,17,0,72,AAA,2013J,TMA,19,10.0
3,1752,32885,26,0,69,AAA,2013J,TMA,19,10.0
4,1752,38053,19,0,79,AAA,2013J,TMA,19,10.0
5,1752,45462,20,0,70,AAA,2013J,TMA,19,10.0
6,1752,45642,18,0,72,AAA,2013J,TMA,19,10.0
7,1752,52130,19,0,72,AAA,2013J,TMA,19,10.0
8,1752,53025,9,0,71,AAA,2013J,TMA,19,10.0
9,1752,57506,18,0,68,AAA,2013J,TMA,19,10.0


In [111]:
assessment_schedule = (
    assessments[
        [
            "code_module",
            "code_presentation",
            "id_assessment",
            "assessment_type",
            "date",
            "weight"
        ]
    ]
    .sort_values(
        [
            "code_module",
            "code_presentation",
            "date"
        ]
    )
)

display(assessment_schedule.head(20))

,code_module,code_presentation,id_assessment,assessment_type,date,weight
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
0,AAA,2013J,1752,TMA,19,10.0
4,AAA,2013J,1756,TMA,215,30.0
1,AAA,2013J,1753,TMA,54,20.0
5,AAA,2013J,1757,Exam,?,100.0
8,AAA,2014J,1760,TMA,117,20.0
9,AAA,2014J,1761,TMA,166,20.0
6,AAA,2014J,1758,TMA,19,10.0
10,AAA,2014J,1762,TMA,215,30.0


In [112]:
aaa_2013j = assessment_schedule[
    (assessment_schedule["code_module"] == "AAA") &
    (assessment_schedule["code_presentation"] == "2013J")
]

display(aaa_2013j)

,code_module,code_presentation,id_assessment,assessment_type,date,weight
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
0,AAA,2013J,1752,TMA,19,10.0
4,AAA,2013J,1756,TMA,215,30.0
1,AAA,2013J,1753,TMA,54,20.0
5,AAA,2013J,1757,Exam,?,100.0


In [113]:
courses[
    (courses["code_module"] == "AAA") &
    (courses["code_presentation"] == "2013J")
]

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268


In [114]:
student_vle.head(10)
print(student_vle.shape)
print(student_vle.columns.tolist())

(10655280, 6)
['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']


In [115]:
aaa_2013j_assessments = assessment_schedule[
    (assessment_schedule["code_module"] == "AAA") &
    (assessment_schedule["code_presentation"] == "2013J")
]

display(aaa_2013j_assessments)

,code_module,code_presentation,id_assessment,assessment_type,date,weight
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
0,AAA,2013J,1752,TMA,19,10.0
4,AAA,2013J,1756,TMA,215,30.0
1,AAA,2013J,1753,TMA,54,20.0
5,AAA,2013J,1757,Exam,?,100.0


In [116]:
assessment_data.groupby(
    ["code_module", "code_presentation"]
)["id_student"].nunique().head(20)
assessment_data["assessment_type"].value_counts()

,count
assessment_type,
TMA,98426
CMA,70527
Exam,4959


In [117]:
assessment_data.groupby(
    ["code_module", "code_presentation", "assessment_type"]
).size().reset_index(name="count")

,code_module,code_presentation,assessment_type,count
0,AAA,2013J,TMA,1633
1,AAA,2014J,TMA,1516
2,BBB,2013B,CMA,5049
3,BBB,2013B,TMA,6207
4,BBB,2013J,CMA,6416
5,BBB,2013J,TMA,7959
6,BBB,2014B,CMA,4493
7,BBB,2014B,TMA,5500
8,BBB,2014J,TMA,7408
9,CCC,2014B,CMA,3920


In [118]:
assessment_data.groupby(
    ["code_module", "code_presentation"]
)["id_student"].nunique().reset_index(name="students")

,code_module,code_presentation,students
0,AAA,2013J,365
1,AAA,2014J,340
2,BBB,2013B,1369
3,BBB,2013J,1710
4,BBB,2014B,1205
5,BBB,2014J,1793
6,CCC,2014B,1415
7,CCC,2014J,1998
8,DDD,2013B,1065
9,DDD,2013J,1507


In [119]:
aaa_2013j_assessments = assessment_schedule[
    (assessment_schedule["code_module"] == "AAA") &
    (assessment_schedule["code_presentation"] == "2013J")
].copy()

display(
    aaa_2013j_assessments[
        ["id_assessment", "assessment_type", "date", "weight"]
    ].sort_values("date", na_position="last")
)

,id_assessment,assessment_type,date,weight
2,1754,TMA,117,20.0
3,1755,TMA,166,20.0
0,1752,TMA,19,10.0
4,1756,TMA,215,30.0
1,1753,TMA,54,20.0
5,1757,Exam,?,100.0


In [120]:
student_id = 11391

student_assessments = assessment_data[
    assessment_data["id_student"] == student_id
].sort_values("date")

display(
    student_assessments[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "id_assessment",
            "assessment_type",
            "date",
            "date_submitted",
            "score",
            "weight"
        ]
    ]
)

,id_student,code_module,code_presentation,id_assessment,assessment_type,date,date_submitted,score,weight
842,11391,AAA,2013J,1754,TMA,117,115,80,20.0
1032,11391,AAA,2013J,1755,TMA,166,164,85,20.0
0,11391,AAA,2013J,1752,TMA,19,18,78,10.0
1457,11391,AAA,2013J,1756,TMA,215,212,82,30.0
501,11391,AAA,2013J,1753,TMA,54,53,85,20.0


In [121]:
student_activity = student_vle[
    student_vle["id_student"] == student_id
].sort_values("date")

display(student_activity.head(20))

,code_module,code_presentation,id_student,id_site,date,sum_click
6758,AAA,2013J,11391,546669,-5,16
6759,AAA,2013J,11391,546662,-5,44
6760,AAA,2013J,11391,546652,-5,1
6761,AAA,2013J,11391,546668,-5,2
6762,AAA,2013J,11391,546652,-5,1
6763,AAA,2013J,11391,546670,-5,2
6764,AAA,2013J,11391,546671,-5,2
6765,AAA,2013J,11391,546614,-5,7
6766,AAA,2013J,11391,546667,-5,10
6767,AAA,2013J,11391,546672,-5,2


In [122]:
print("Assessment records:", len(student_assessments))
print("VLE records:", len(student_activity))
print("Total clicks:", student_activity["sum_click"].sum())

Assessment records: 5
VLE records: 196
Total clicks: 934


In [123]:
student_activity_positive = student_activity[
    student_activity["date"] >= 0
].copy()

student_activity_positive["Period"] = pd.cut(
    student_activity_positive["date"],
    bins=[-1, 66, 133, 200, 267],
    labels=["Period 1", "Period 2", "Period 3", "Period 4"]
)

activity_by_period = (
    student_activity_positive
    .groupby("Period", observed=False)["sum_click"]
    .sum()
    .reset_index()
)

display(activity_by_period)

,Period,sum_click
0,Period 1,447
1,Period 2,165
2,Period 3,38
3,Period 4,186


In [124]:
# Make a copy
student_assessments_positive = student_assessments.copy()

# Convert date and score to numeric
student_assessments_positive["date"] = pd.to_numeric(
    student_assessments_positive["date"],
    errors="coerce"
)

student_assessments_positive["score"] = pd.to_numeric(
    student_assessments_positive["score"],
    errors="coerce"
)

# Keep only assessments with valid dates
student_assessments_positive = student_assessments_positive[
    student_assessments_positive["date"] >= 0
].copy()

# Create four chronological periods
student_assessments_positive["Period"] = pd.cut(
    student_assessments_positive["date"],
    bins=[-1, 66, 133, 200, 267],
    labels=["Period 1", "Period 2", "Period 3", "Period 4"]
)

# Calculate average score and assessment count
assessment_by_period = (
    student_assessments_positive
    .groupby("Period", observed=False)
    .agg(
        assessment_score=("score", "mean"),
        assessment_count=("score", "count")
    )
    .reset_index()
)

display(assessment_by_period)

,Period,assessment_score,assessment_count
0,Period 1,81.5,2
1,Period 2,80.0,1
2,Period 3,85.0,1
3,Period 4,82.0,1


In [125]:
student_features = activity_by_period.merge(
    assessment_by_period,
    on="Period",
    how="left"
)

display(student_features)

,Period,sum_click,assessment_score,assessment_count
0,Period 1,447,81.5,2
1,Period 2,165,80.0,1
2,Period 3,38,85.0,1
3,Period 4,186,82.0,1


In [126]:
assessment_schedule.groupby(
    ["code_module", "code_presentation"]
)["date"].apply(
    lambda x: pd.to_numeric(x, errors="coerce").notna().sum()
).reset_index(name="dated_assessments")

,code_module,code_presentation,dated_assessments
0,AAA,2013J,5
1,AAA,2014J,5
2,BBB,2013B,11
3,BBB,2013J,11
4,BBB,2014B,11
5,BBB,2014J,5
6,CCC,2014B,8
7,CCC,2014J,8
8,DDD,2013B,14
9,DDD,2013J,7


In [127]:
assessment_schedule.groupby(
    ["code_module", "code_presentation"]
)["date"].apply(
    lambda x: pd.to_numeric(x, errors="coerce").max()
).reset_index(name="last_assessment_day")

,code_module,code_presentation,last_assessment_day
0,AAA,2013J,215.0
1,AAA,2014J,215.0
2,BBB,2013B,187.0
3,BBB,2013J,208.0
4,BBB,2014B,194.0
5,BBB,2014J,201.0
6,CCC,2014B,207.0
7,CCC,2014J,214.0
8,DDD,2013B,240.0
9,DDD,2013J,261.0


In [128]:
student_assessments_sorted = (
    student_assessments_positive
    .sort_values("date")
    .reset_index(drop=True)
)

student_assessments_sorted["assessment_number"] = (
    student_assessments_sorted.index + 1
)

student_assessments_sorted["previous_average"] = (
    student_assessments_sorted["score"]
    .shift(1)
    .expanding()
    .mean()
)

display(
    student_assessments_sorted[
        [
            "assessment_number",
            "date",
            "assessment_type",
            "score",
            "previous_average"
        ]
    ]
)

,assessment_number,date,assessment_type,score,previous_average
0,1,19,TMA,78,NaN
1,2,54,TMA,85,78.0
2,3,117,TMA,80,81.5
3,4,166,TMA,85,81.0
4,5,215,TMA,82,82.0


In [129]:
# Make sure VLE date is numeric
student_activity["date"] = pd.to_numeric(
    student_activity["date"],
    errors="coerce"
)

# Sort activity chronologically
student_activity = student_activity.sort_values("date").reset_index(drop=True)

# Calculate cumulative clicks up to each point
student_activity["cumulative_clicks"] = (
    student_activity["sum_click"].cumsum()
)

# Get the assessment dates
assessment_dates = student_assessments_sorted[
    ["assessment_number", "date", "score"]
].copy()

# Calculate total clicks BEFORE each assessment
activity_before_assessment = []

for _, assessment in assessment_dates.iterrows():

    assessment_day = assessment["date"]

    clicks_before = student_activity[
        student_activity["date"] < assessment_day
    ]["sum_click"].sum()

    activity_before_assessment.append({
        "assessment_number": assessment["assessment_number"],
        "date": assessment_day,
        "score": assessment["score"],
        "clicks_before": clicks_before
    })

activity_before_assessment = pd.DataFrame(
    activity_before_assessment
)

display(activity_before_assessment)

,assessment_number,date,score,clicks_before
0,1,19,78,401
1,2,54,85,528
2,3,117,80,628
3,4,166,85,713
4,5,215,82,791


In [130]:
student_model_data = assessment_data_sorted = (
    student_assessments_sorted[
        [
            "assessment_number",
            "date",
            "assessment_type",
            "score",
            "previous_average"
        ]
    ]
    .merge(
        activity_before_assessment[
            [
                "assessment_number",
                "clicks_before"
            ]
        ],
        on="assessment_number",
        how="left"
    )
)

display(student_model_data)

,assessment_number,date,assessment_type,score,previous_average,clicks_before
0,1,19,TMA,78,NaN,401
1,2,54,TMA,85,78.0,528
2,3,117,TMA,80,81.5,628
3,4,166,TMA,85,81.0,713
4,5,215,TMA,82,82.0,791


In [131]:
# Start from chronological assessment data
model_data = assessment_data_sorted.copy()

# Create previous assessment scores
model_data["previous_score_1"] = model_data["score"].shift(1)
model_data["previous_score_2"] = model_data["score"].shift(2)
model_data["previous_score_3"] = model_data["score"].shift(3)
model_data["previous_score_4"] = model_data["score"].shift(4)

display(
    model_data[
        [
            "assessment_number",
            "date",
            "previous_score_1",
            "previous_score_2",
            "previous_score_3",
            "previous_score_4",
            "previous_average",
            "clicks_before",
            "score"
        ]
    ]
)

,assessment_number,date,previous_score_1,previous_score_2,previous_score_3,previous_score_4,previous_average,clicks_before,score
0,1,19,NaN,NaN,NaN,NaN,NaN,401,78
1,2,54,78.0,NaN,NaN,NaN,78.0,528,85
2,3,117,85.0,78.0,NaN,NaN,81.5,628,80
3,4,166,80.0,85.0,78.0,NaN,81.0,713,85
4,5,215,85.0,80.0,85.0,78.0,82.0,791,82


In [132]:
# Keep only rows where all 4 previous scores are available
training_examples = model_data.dropna(
    subset=[
        "previous_score_1",
        "previous_score_2",
        "previous_score_3",
        "previous_score_4"
    ]
).copy()

display(training_examples)

,assessment_number,date,assessment_type,score,previous_average,clicks_before,previous_score_1,previous_score_2,previous_score_3,previous_score_4
4,5,215,TMA,82,82.0,791,85.0,80.0,85.0,78.0


In [133]:
# Make a fresh copy
full_assessment = assessment_data.copy()

# Convert important columns to numeric
full_assessment["date"] = pd.to_numeric(
    full_assessment["date"],
    errors="coerce"
)

full_assessment["score"] = pd.to_numeric(
    full_assessment["score"],
    errors="coerce"
)

# Remove records where date or score is unavailable
full_assessment = full_assessment.dropna(
    subset=["date", "score"]
).copy()

# We will use dated assessments only
full_assessment = full_assessment[
    full_assessment["date"] >= 0
].copy()

# Sort chronologically within each student/course
full_assessment = full_assessment.sort_values(
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
).reset_index(drop=True)

print("Total assessment records:", len(full_assessment))

display(
    full_assessment[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "date",
            "assessment_type",
            "score"
        ]
    ].head(20)
)

Total assessment records: 170874


,id_student,code_module,code_presentation,date,assessment_type,score
0,6516,AAA,2014J,19.0,TMA,60.0
1,6516,AAA,2014J,54.0,TMA,48.0
2,6516,AAA,2014J,117.0,TMA,63.0
3,6516,AAA,2014J,166.0,TMA,61.0
4,6516,AAA,2014J,215.0,TMA,77.0
5,8462,DDD,2013J,25.0,TMA,93.0
6,8462,DDD,2013J,53.0,TMA,83.0
7,8462,DDD,2013J,88.0,TMA,87.0
8,8462,DDD,2014J,20.0,TMA,93.0
9,8462,DDD,2014J,41.0,TMA,83.0


In [134]:
group_columns = [
    "id_student",
    "code_module",
    "code_presentation"
]

full_assessment["previous_score_1"] = (
    full_assessment
    .groupby(group_columns)["score"]
    .shift(1)
)

full_assessment["previous_score_2"] = (
    full_assessment
    .groupby(group_columns)["score"]
    .shift(2)
)

full_assessment["previous_score_3"] = (
    full_assessment
    .groupby(group_columns)["score"]
    .shift(3)
)

full_assessment["previous_score_4"] = (
    full_assessment
    .groupby(group_columns)["score"]
    .shift(4)
)

In [135]:
previous_scores = [
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "previous_score_4"
]

full_assessment["previous_average"] = (
    full_assessment[previous_scores]
    .mean(axis=1)
)

In [136]:
training_data = full_assessment.dropna(
    subset=[
        "previous_score_1",
        "previous_score_2",
        "previous_score_3",
        "previous_score_4"
    ]
).copy()

print("Total assessment records:", len(full_assessment))
print("Valid training examples:", len(training_data))

Total assessment records: 170874
Valid training examples: 80593


In [137]:
student_check = training_data[
    training_data["id_student"] == 11391
]

display(
    student_check[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "date",
            "previous_score_1",
            "previous_score_2",
            "previous_score_3",
            "previous_score_4",
            "previous_average",
            "score"
        ]
    ]
)

,id_student,code_module,code_presentation,date,previous_score_1,previous_score_2,previous_score_3,previous_score_4,previous_average,score
16,11391,AAA,2013J,215.0,85.0,80.0,85.0,78.0,82.0,82.0


In [138]:
print("Total assessment records:", len(full_assessment))
print("Valid training examples:", len(training_data))

Total assessment records: 170874
Valid training examples: 80593


In [139]:
# Make a copy
vle_features = student_vle.copy()

# Convert date and clicks to numeric
vle_features["date"] = pd.to_numeric(
    vle_features["date"],
    errors="coerce"
)

vle_features["sum_click"] = pd.to_numeric(
    vle_features["sum_click"],
    errors="coerce"
)

# Remove invalid rows
vle_features = vle_features.dropna(
    subset=["date", "sum_click"]
).copy()

# Keep only activity from course start onward
vle_features = vle_features[
    vle_features["date"] >= 0
].copy()

print("Valid VLE records:", len(vle_features))

Valid VLE records: 9966292


In [140]:
daily_activity = (
    vle_features
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation",
            "date"
        ],
        as_index=False
    )["sum_click"]
    .sum()
    .rename(columns={"sum_click": "daily_clicks"})
)

print("Daily activity records:", len(daily_activity))

display(daily_activity.head(10))

Daily activity records: 1700367


,id_student,code_module,code_presentation,date,daily_clicks
0,6516,AAA,2014J,0,71
1,6516,AAA,2014J,1,45
2,6516,AAA,2014J,2,60
3,6516,AAA,2014J,3,17
4,6516,AAA,2014J,5,23
5,6516,AAA,2014J,6,13
6,6516,AAA,2014J,9,3
7,6516,AAA,2014J,10,7
8,6516,AAA,2014J,11,22
9,6516,AAA,2014J,13,10


In [141]:
daily_activity = daily_activity.sort_values(
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
).reset_index(drop=True)

daily_activity["cumulative_clicks"] = (
    daily_activity
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )["daily_clicks"]
    .cumsum()
)

In [142]:
assessment_for_activity = training_data[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
].copy()

daily_activity_for_merge = daily_activity.copy()

# Force date to numeric integer
assessment_for_activity["date"] = pd.to_numeric(
    assessment_for_activity["date"],
    errors="coerce"
).astype("int64")

daily_activity_for_merge["date"] = pd.to_numeric(
    daily_activity_for_merge["date"],
    errors="coerce"
).astype("int64")

# Make sure IDs have consistent string types
assessment_for_activity["code_module"] = (
    assessment_for_activity["code_module"].astype(str)
)

assessment_for_activity["code_presentation"] = (
    assessment_for_activity["code_presentation"].astype(str)
)

daily_activity_for_merge["code_module"] = (
    daily_activity_for_merge["code_module"].astype(str)
)

daily_activity_for_merge["code_presentation"] = (
    daily_activity_for_merge["code_presentation"].astype(str)
)

assessment_for_activity["id_student"] = (
    assessment_for_activity["id_student"].astype("int64")
)

daily_activity_for_merge["id_student"] = (
    daily_activity_for_merge["id_student"].astype("int64")
)

In [143]:
activity_results = []

presentation_groups = assessment_for_activity[
    [
        "code_module",
        "code_presentation"
    ]
].drop_duplicates()

for _, presentation in presentation_groups.iterrows():

    module = presentation["code_module"]
    presentation_code = presentation["code_presentation"]

    # Assessment rows for this presentation
    assess_group = assessment_for_activity[
        (assessment_for_activity["code_module"] == module) &
        (assessment_for_activity["code_presentation"] == presentation_code)
    ].copy()

    # VLE rows for this presentation
    vle_group = daily_activity_for_merge[
        (daily_activity_for_merge["code_module"] == module) &
        (daily_activity_for_merge["code_presentation"] == presentation_code)
    ].copy()

    # Sort ONLY by date for merge_asof
    assess_group = assess_group.sort_values(
        ["date", "id_student"]
    ).reset_index(drop=True)

    vle_group = vle_group.sort_values(
        ["date", "id_student"]
    ).reset_index(drop=True)

    # Match latest VLE date BEFORE assessment date
    if len(vle_group) > 0:

        merged_group = pd.merge_asof(
            assess_group,
            vle_group[
                [
                    "id_student",
                    "date",
                    "daily_clicks",
                    "cumulative_clicks"
                ]
            ],
            on="date",
            by="id_student",
            direction="backward",
            allow_exact_matches=False
        )

        activity_results.append(merged_group)

# Combine all presentations
assessment_with_activity = pd.concat(
    activity_results,
    ignore_index=True
)

print("Merge successful!")
print("Rows:", len(assessment_with_activity))

Merge successful!
Rows: 80593


In [144]:
display(
    assessment_with_activity[
        assessment_with_activity["id_student"] == 11391
    ].sort_values("date")
)

,id_student,code_module,code_presentation,date,daily_clicks,cumulative_clicks
273,11391,AAA,2013J,215,43.0,693.0


In [145]:
# Prepare daily activity for cumulative calculation
daily_activity = daily_activity.sort_values(
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
).reset_index(drop=True)

# Create cumulative clicks within each student/course/presentation
daily_activity["cumulative_clicks"] = (
    daily_activity
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )["daily_clicks"]
    .cumsum()
)

In [146]:
# Keep only the information needed
activity_lookup = daily_activity[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date",
        "cumulative_clicks"
    ]
].copy()

# Rename for clarity
activity_lookup = activity_lookup.rename(
    columns={"date": "activity_date"}
)

# Sort
activity_lookup = activity_lookup.sort_values(
    [
        "id_student",
        "code_module",
        "code_presentation",
        "activity_date"
    ]
)

In [147]:
assessment_activity = training_data[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
].copy()

assessment_activity["group_key"] = (
    assessment_activity["id_student"].astype(str)
    + "_"
    + assessment_activity["code_module"].astype(str)
    + "_"
    + assessment_activity["code_presentation"].astype(str)
)

activity_lookup["group_key"] = (
    activity_lookup["id_student"].astype(str)
    + "_"
    + activity_lookup["code_module"].astype(str)
    + "_"
    + activity_lookup["code_presentation"].astype(str)
)

In [148]:
assessment_activity = assessment_activity.sort_values(
    ["group_key", "date"]
).reset_index(drop=True)

activity_lookup = activity_lookup.sort_values(
    ["group_key", "activity_date"]
).reset_index(drop=True)

In [149]:
# ---------------------------------------------------------
# Calculate clicks BEFORE each assessment
# ---------------------------------------------------------

activity_source = daily_activity.copy()

# Make sure dates are integers
activity_source["date"] = pd.to_numeric(
    activity_source["date"],
    errors="coerce"
).astype("int64")

activity_source["daily_clicks"] = pd.to_numeric(
    activity_source["daily_clicks"],
    errors="coerce"
).fillna(0)

# Sort activity
activity_source = activity_source.sort_values(
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
).reset_index(drop=True)

# Assessment rows
assessment_source = training_data[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ]
].copy()

assessment_source["date"] = pd.to_numeric(
    assessment_source["date"],
    errors="coerce"
).astype("int64")

# ---------------------------------------------------------
# Calculate cumulative clicks
# ---------------------------------------------------------

activity_source["cumulative_clicks"] = (
    activity_source
    .groupby(
        [
            "id_student",
            "code_module",
            "code_presentation"
        ]
    )["daily_clicks"]
    .cumsum()
)

# ---------------------------------------------------------
# Create a lookup dictionary
# ---------------------------------------------------------

activity_lookup = {}

for key, group in activity_source.groupby(
    [
        "id_student",
        "code_module",
        "code_presentation"
    ]
):

    activity_lookup[key] = (
        group["date"].to_numpy(),
        group["cumulative_clicks"].to_numpy()
    )

# ---------------------------------------------------------
# Calculate clicks before assessment
# ---------------------------------------------------------

clicks_before = []

for _, row in assessment_source.iterrows():

    key = (
        row["id_student"],
        row["code_module"],
        row["code_presentation"]
    )

    if key not in activity_lookup:
        clicks_before.append(0)
        continue

    activity_dates, cumulative_values = activity_lookup[key]

    # Number of activity dates strictly before assessment
    position = activity_dates.searchsorted(
        row["date"],
        side="left"
    )

    if position == 0:
        clicks_before.append(0)
    else:
        clicks_before.append(
            cumulative_values[position - 1]
        )

assessment_source["clicks_before"] = clicks_before

print("Rows:", len(assessment_source))

display(
    assessment_source[
        assessment_source["id_student"] == 11391
    ].sort_values("date")
)

Rows: 80593


,id_student,code_module,code_presentation,date,clicks_before
16,11391,AAA,2013J,215,693


In [150]:
student_11391_vle = daily_activity[
    (daily_activity["id_student"] == 11391) &
    (daily_activity["code_module"] == "AAA") &
    (daily_activity["code_presentation"] == "2013J")
].sort_values("date")

display(
    student_11391_vle[
        student_11391_vle["date"] < 215
    ][
        ["date", "daily_clicks", "cumulative_clicks"]
    ]
)

print(
    "Direct clicks before day 215:",
    student_11391_vle[
        student_11391_vle["date"] < 215
    ]["daily_clicks"].sum()
)

,date,daily_clicks,cumulative_clicks
202,0,49,49
203,1,127,176
204,2,4,180
205,6,3,183
206,9,20,203
207,17,51,254
208,18,49,303
209,29,18,321
210,30,5,326
211,31,3,329


Direct clicks before day 215: 693


In [151]:
# Add a unique row identifier before merging
training_data = training_data.reset_index(drop=True)
training_data["row_id"] = training_data.index

assessment_source = assessment_source.reset_index(drop=True)

# Merge clicks_before into training data
training_data = training_data.merge(
    assessment_source[
        [
            "id_student",
            "code_module",
            "code_presentation",
            "date",
            "clicks_before"
        ]
    ],
    on=[
        "id_student",
        "code_module",
        "code_presentation",
        "date"
    ],
    how="left"
)

print("Training rows:", len(training_data))
print(
    "Missing clicks_before:",
    training_data["clicks_before"].isna().sum()
)

Training rows: 292621
Missing clicks_before: 0


In [152]:
print("Training rows:", len(training_data))

print("\nColumns:")
print(training_data.columns.tolist())

print("\nMissing values:")
print(
    training_data[
        [
            "previous_score_1",
            "previous_score_2",
            "previous_score_3",
            "previous_score_4",
            "previous_average",
            "clicks_before",
            "score"
        ]
    ].isna().sum()
)

print("\nStudent 11391:")
display(
    training_data[
        training_data["id_student"] == 11391
    ][
        [
            "date",
            "previous_score_1",
            "previous_score_2",
            "previous_score_3",
            "previous_score_4",
            "previous_average",
            "clicks_before",
            "score"
        ]
    ]
)

Training rows: 292621

Columns:
['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight', 'previous_score_1', 'previous_score_2', 'previous_score_3', 'previous_score_4', 'previous_average', 'row_id', 'clicks_before']

Missing values:
previous_score_1    0
previous_score_2    0
previous_score_3    0
previous_score_4    0
previous_average    0
clicks_before       0
score               0
dtype: int64

Student 11391:


,date,previous_score_1,previous_score_2,previous_score_3,previous_score_4,previous_average,clicks_before,score
1,215.0,85.0,80.0,85.0,78.0,82.0,693,82.0


In [153]:
training_data["performance_trend"] = (
    training_data["previous_score_1"]
    - training_data["previous_score_4"]
)

In [154]:
training_data["performance_trend"] = (
    training_data["previous_score_1"]
    - training_data["previous_score_4"]
)

display(
    training_data[
        [
            "previous_score_1",
            "previous_score_2",
            "previous_score_3",
            "previous_score_4",
            "previous_average",
            "performance_trend",
            "clicks_before",
            "score"
        ]
    ].head(10)
)

,previous_score_1,previous_score_2,previous_score_3,previous_score_4,previous_average,performance_trend,clicks_before,score
0,61.0,63.0,48.0,60.0,58.00,1.0,2214,77.0
1,85.0,80.0,85.0,78.0,82.00,7.0,693,82.0
2,60.0,93.0,94.0,78.0,81.25,-18.0,531,73.0
3,73.0,60.0,93.0,94.0,80.00,-21.0,561,69.0
4,69.0,73.0,60.0,93.0,73.75,-24.0,609,67.0
5,67.0,69.0,73.0,60.0,67.25,7.0,726,56.0
6,89.0,100.0,90.0,90.0,92.25,-1.0,271,100.0
7,100.0,89.0,100.0,90.0,94.75,10.0,345,88.0
8,100.0,89.0,100.0,90.0,94.75,10.0,345,88.0
9,88.0,100.0,89.0,100.0,94.25,-12.0,345,100.0


In [155]:
from sklearn.model_selection import GroupShuffleSplit

features = [
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "previous_score_4",
    "previous_average",
    "performance_trend",
    "clicks_before"
]

X = training_data[features]
y = training_data["score"]

groups = training_data["id_student"]

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique students:", groups.nunique())

Features: ['previous_score_1', 'previous_score_2', 'previous_score_3', 'previous_score_4', 'previous_average', 'performance_trend', 'clicks_before']
X shape: (292621, 7)
y shape: (292621,)
Unique students: 15956


In [156]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training students:",
    training_data.iloc[train_idx]["id_student"].nunique()
)

print(
    "Testing students:",
    training_data.iloc[test_idx]["id_student"].nunique()
)

Training rows: 234692
Testing rows: 57929
Training students: 12764
Testing students: 3192


In [157]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [158]:
y_pred = model.predict(X_test)

print("Predictions generated:", len(y_pred))

print("\nFirst 10 predictions:")
print(y_pred[:10])

Predictions generated: 57929

First 10 predictions:
[67.16152831 92.14259319 85.34979211 85.34979211 94.40838424 94.40838424
 86.13620763 86.13620763 94.51272634 94.51272634]


In [159]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("Random Forest Performance")
print("-------------------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

Random Forest Performance
-------------------------
MAE  : 9.24
RMSE : 12.74
R²   : 0.3371


In [160]:
baseline_pred = X_test["previous_average"]

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

print("Previous Average Baseline")
print("-------------------------")
print(f"MAE  : {baseline_mae:.2f}")
print(f"RMSE : {baseline_rmse:.2f}")
print(f"R²   : {baseline_r2:.4f}")

Previous Average Baseline
-------------------------
MAE  : 9.95
RMSE : 13.74
R²   : 0.2282


In [161]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

display(feature_importance)

,Feature,Importance
4,previous_average,0.543282
6,clicks_before,0.128767
0,previous_score_1,0.124117
1,previous_score_2,0.081126
2,previous_score_3,0.050415
5,performance_trend,0.041149
3,previous_score_4,0.031144


In [162]:
import joblib

model_path = "/content/student_performance_model.pkl"

joblib.dump(model, model_path)

print("Model saved successfully:")
print(model_path)

Model saved successfully:
/content/student_performance_model.pkl


In [163]:
feature_info = {
    "features": features
}

feature_path = "/content/student_performance_features.pkl"

joblib.dump(feature_info, feature_path)

print("Feature list saved successfully:")
print(feature_path)

Feature list saved successfully:
/content/student_performance_features.pkl


In [164]:
def predict_student_performance(
    previous_score_1,
    previous_score_2,
    previous_score_3,
    previous_score_4,
    clicks_before
):
    previous_average = (
        previous_score_1
        + previous_score_2
        + previous_score_3
        + previous_score_4
    ) / 4

    performance_trend = (
        previous_score_1
        - previous_score_4
    )

    input_data = pd.DataFrame([{
        "previous_score_1": previous_score_1,
        "previous_score_2": previous_score_2,
        "previous_score_3": previous_score_3,
        "previous_score_4": previous_score_4,
        "previous_average": previous_average,
        "performance_trend": performance_trend,
        "clicks_before": clicks_before
    }])

    prediction = model.predict(
        input_data[features]
    )[0]

    return prediction

In [165]:
prediction = predict_student_performance(
    previous_score_1=85,
    previous_score_2=80,
    previous_score_3=85,
    previous_score_4=78,
    clicks_before=693
)

print(f"Predicted next score: {prediction:.2f}")

Predicted next score: 82.81


In [166]:
student_features = [
    "previous_score_1",
    "previous_score_2",
    "previous_score_3",
    "previous_score_4",
    "previous_average",
    "performance_trend"
]

X_student = training_data[student_features]
y_student = training_data["score"]

groups_student = training_data["id_student"]

print("Features:")
for feature in student_features:
    print("-", feature)

print("\nDataset shape:", X_student.shape)

Features:
- previous_score_1
- previous_score_2
- previous_score_3
- previous_score_4
- previous_average
- performance_trend

Dataset shape: (292621, 6)


In [167]:
X_student_train = X_student.iloc[train_idx]
X_student_test = X_student.iloc[test_idx]

y_student_train = y_student.iloc[train_idx]
y_student_test = y_student.iloc[test_idx]

print("Training rows:", len(X_student_train))
print("Testing rows:", len(X_student_test))

Training rows: 234692
Testing rows: 57929


In [168]:
from sklearn.ensemble import RandomForestRegressor

student_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

student_model.fit(
    X_student_train,
    y_student_train
)

print("Student-friendly Random Forest training completed.")

Student-friendly Random Forest training completed.


In [169]:
student_pred = student_model.predict(X_student_test)

student_mae = mean_absolute_error(
    y_student_test,
    student_pred
)

student_rmse = np.sqrt(
    mean_squared_error(
        y_student_test,
        student_pred
    )
)

student_r2 = r2_score(
    y_student_test,
    student_pred
)

print("Student-Friendly Random Forest")
print("--------------------------------")
print(f"MAE  : {student_mae:.2f}")
print(f"RMSE : {student_rmse:.2f}")
print(f"R²   : {student_r2:.4f}")

Student-Friendly Random Forest
--------------------------------
MAE  : 9.31
RMSE : 12.88
R²   : 0.3223


In [170]:
final_model_path = "/content/student_performance_model.pkl"

joblib.dump(
    student_model,
    final_model_path
)

print("Final model saved:")
print(final_model_path)

Final model saved:
/content/student_performance_model.pkl


In [171]:
final_features_path = "/content/student_performance_features.pkl"

joblib.dump(
    student_features,
    final_features_path
)

print("Final feature list saved:")
print(final_features_path)

Final feature list saved:
/content/student_performance_features.pkl


In [172]:
def predict_student_performance(
    performance_1,
    performance_2,
    performance_3,
    performance_4
):
    # Most recent performance first
    previous_score_1 = performance_4
    previous_score_2 = performance_3
    previous_score_3 = performance_2
    previous_score_4 = performance_1

    previous_average = (
        previous_score_1
        + previous_score_2
        + previous_score_3
        + previous_score_4
    ) / 4

    performance_trend = (
        previous_score_1
        - previous_score_4
    )

    input_data = pd.DataFrame([{
        "previous_score_1": previous_score_1,
        "previous_score_2": previous_score_2,
        "previous_score_3": previous_score_3,
        "previous_score_4": previous_score_4,
        "previous_average": previous_average,
        "performance_trend": performance_trend
    }])

    prediction = student_model.predict(
        input_data[student_features]
    )[0]

    return prediction

In [184]:
%%writefile /content/app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Student Progress Tracker",
    page_icon="🎓",
    layout="wide"
)


# ============================================================
# LOAD TRAINED ML MODEL
# ============================================================

MODEL_PATH = "/content/drive/MyDrive/StudentProgressTracker/models/student_performance_model.pkl"

FEATURES_PATH = "/content/drive/MyDrive/StudentProgressTracker/models/student_performance_features.pkl"

@st.cache_resource
def load_model():
    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURES_PATH)

    return model, features


try:
    model, model_features = load_model()

except Exception as e:
    st.error("Unable to load the trained ML model.")
    st.error(str(e))
    st.stop()


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def predict_student_performance(
    performance_1,
    performance_2,
    performance_3,
    performance_4
):
    """
    Predict the next performance score using
    the trained student-friendly Random Forest model.

    Performance 1 = oldest
    Performance 4 = most recent
    """

    # Convert user's chronological input into
    # the order expected by the ML model.

    previous_score_1 = performance_4
    previous_score_2 = performance_3
    previous_score_3 = performance_2
    previous_score_4 = performance_1

    # Average of previous performances
    previous_average = (
        previous_score_1
        + previous_score_2
        + previous_score_3
        + previous_score_4
    ) / 4

    # Recent performance compared with oldest performance
    performance_trend = (
        previous_score_1
        - previous_score_4
    )

    input_data = pd.DataFrame([{
        "previous_score_1": previous_score_1,
        "previous_score_2": previous_score_2,
        "previous_score_3": previous_score_3,
        "previous_score_4": previous_score_4,
        "previous_average": previous_average,
        "performance_trend": performance_trend
    }])

    # Make sure features are in exactly the same
    # order used during model training.
    input_data = input_data[model_features]

    prediction = model.predict(input_data)[0]

    # Keep prediction between 0 and 100
    prediction = np.clip(prediction, 0, 100)

    return float(prediction)


def get_category(score):

    if score >= 85:
        return "Excellent"
    elif score >= 70:
        return "Good"
    elif score >= 50:
        return "Average"
    else:
        return "At Risk"


def get_trend(performance_1, performance_2,
              performance_3, performance_4):

    first_half = (
        performance_1 + performance_2
    ) / 2

    second_half = (
        performance_3 + performance_4
    ) / 2

    difference = second_half - first_half

    if difference >= 5:
        return "Improving"

    elif difference <= -5:
        return "Declining"

    else:
        return "Stable"


def get_recommendation(category, trend):

    if category == "Excellent":
        if trend == "Improving":
            return "Excellent progress. Continue the current study strategy and maintain consistency."
        elif trend == "Declining":
            return "Your current performance is strong, but the recent decline should be monitored."
        else:
            return "Performance is strong and stable. Continue your current approach."

    elif category == "Good":
        if trend == "Improving":
            return "Good performance with positive progress. Continue your current study routine."
        elif trend == "Declining":
            return "Performance is still good, but the recent decline needs attention."
        else:
            return "Performance is stable. Focus on consistent practice to move toward excellent performance."

    elif category == "Average":
        if trend == "Improving":
            return "You are improving. Increase practice time and continue focusing on weaker subjects."
        elif trend == "Declining":
            return "Performance is declining. Review difficult topics and increase regular practice."
        else:
            return "Performance is stable but can improve. Focus on consistency and targeted practice."

    else:
        if trend == "Improving":
            return "You are showing improvement, but additional practice is recommended to reach a safer performance level."
        else:
            return "Immediate attention is recommended. Review weak areas, increase practice, and seek academic support if needed."


# ============================================================
# HEADER
# ============================================================

st.title("🎓 Student Progress Tracking & Performance Prediction")

st.write(
    "Analyze historical academic performance and predict the student's "
    "next performance using a trained Random Forest machine-learning model."
)

st.divider()


# ============================================================
# STUDENT INFORMATION
# ============================================================

st.subheader("👨‍🎓 Student Information")

col1, col2 = st.columns(2)

with col1:

    student_name = st.text_input(
        "Student Name",
        placeholder="Enter student name"
    )

    student_id = st.text_input(
        "Student ID",
        placeholder="Enter student ID"
    )

    course = st.text_input(
        "Course",
        placeholder="e.g. B.Tech Computer Science"
    )


with col2:

    department = st.text_input(
        "Department",
        placeholder="e.g. Computer Engineering"
    )

    semester = st.selectbox(
        "Semester",
        [
            "Semester 1",
            "Semester 2",
            "Semester 3",
            "Semester 4",
            "Semester 5",
            "Semester 6",
            "Semester 7",
            "Semester 8"
        ]
    )

    attendance = st.number_input(
        "Overall Attendance (%)",
        min_value=0.0,
        max_value=100.0,
        value=75.0,
        step=1.0
    )

    study_hours = st.number_input(
        "Average Study Hours / Day",
        min_value=0.0,
        max_value=24.0,
        value=2.0,
        step=0.5
    )


st.divider()


# ============================================================
# SUBJECT INPUT
# ============================================================

st.subheader("📚 Subject Performance")

st.info(
    "Enter the student's four most recent performance periods. "
    "These values are used by the trained Random Forest model "
    "to predict the next performance."
)

num_subjects = st.number_input(
    "Number of Subjects",
    min_value=1,
    max_value=12,
    value=3,
    step=1
)


subjects = []

for i in range(int(num_subjects)):

    st.markdown(f"### Subject {i + 1}")

    col1, col2, col3, col4, col5 = st.columns(5)

    with col1:
        subject_name = st.text_input(
            "Subject Name",
            key=f"subject_name_{i}",
            placeholder="e.g. Mathematics"
        )

    with col2:
        performance_1 = st.number_input(
            "Period 1",
            min_value=0.0,
            max_value=100.0,
            value=0.0,
            step=1.0,
            key=f"performance_1_{i}"
        )

    with col3:
        performance_2 = st.number_input(
            "Period 2",
            min_value=0.0,
            max_value=100.0,
            value=0.0,
            step=1.0,
            key=f"performance_2_{i}"
        )

    with col4:
        performance_3 = st.number_input(
            "Period 3",
            min_value=0.0,
            max_value=100.0,
            value=0.0,
            step=1.0,
            key=f"performance_3_{i}"
        )

    with col5:
        performance_4 = st.number_input(
            "Period 4",
            min_value=0.0,
            max_value=100.0,
            value=0.0,
            step=1.0,
            key=f"performance_4_{i}"
        )

    subjects.append({
        "Subject": subject_name,
        "Period 1": performance_1,
        "Period 2": performance_2,
        "Period 3": performance_3,
        "Period 4": performance_4
    })


st.divider()


# ============================================================
# ANALYZE BUTTON
# ============================================================

if st.button(
    "🔮 Analyze Student",
    type="primary",
    use_container_width=True
):

    # --------------------------------------------------------
    # Basic validation
    # --------------------------------------------------------

    valid_subjects = [
        subject for subject in subjects
        if subject["Subject"].strip()
    ]

    if not valid_subjects:

        st.warning(
            "Please enter at least one subject name."
        )

        st.stop()


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    results = []

    for subject in valid_subjects:

        predicted_score = predict_student_performance(
            subject["Period 1"],
            subject["Period 2"],
            subject["Period 3"],
            subject["Period 4"]
        )

        category = get_category(
            predicted_score
        )

        trend = get_trend(
            subject["Period 1"],
            subject["Period 2"],
            subject["Period 3"],
            subject["Period 4"]
        )

        recommendation = get_recommendation(
            category,
            trend
        )

        average = (
            subject["Period 1"]
            + subject["Period 2"]
            + subject["Period 3"]
            + subject["Period 4"]
        ) / 4

        results.append({
            "Subject": subject["Subject"],
            "Period 1": subject["Period 1"],
            "Period 2": subject["Period 2"],
            "Period 3": subject["Period 3"],
            "Period 4": subject["Period 4"],
            "Historical Average": average,
            "Predicted Score": predicted_score,
            "Category": category,
            "Trend": trend,
            "Recommendation": recommendation
        })


    results_df = pd.DataFrame(results)


    # ========================================================
    # RESULTS
    # ========================================================

    st.header("📊 Prediction Results")


    # --------------------------------------------------------
    # Overall prediction
    # --------------------------------------------------------

    overall_prediction = results_df[
        "Predicted Score"
    ].mean()

    overall_category = get_category(
        overall_prediction
    )


    # Overall trend
    overall_period_1 = results_df["Period 1"].mean()
    overall_period_2 = results_df["Period 2"].mean()
    overall_period_3 = results_df["Period 3"].mean()
    overall_period_4 = results_df["Period 4"].mean()

    overall_trend = get_trend(
        overall_period_1,
        overall_period_2,
        overall_period_3,
        overall_period_4
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "Predicted Performance",
            f"{overall_prediction:.2f}"
        )

    with col2:
        st.metric(
            "Performance Category",
            overall_category
        )

    with col3:
        st.metric(
            "Overall Trend",
            overall_trend
        )

    with col4:
        st.metric(
            "Attendance",
            f"{attendance:.1f}%"
        )


    st.divider()


    # --------------------------------------------------------
    # Subject results
    # --------------------------------------------------------

    st.subheader("📚 Subject-wise Results")

    display_df = results_df[
        [
            "Subject",
            "Historical Average",
            "Predicted Score",
            "Category",
            "Trend"
        ]
    ].copy()

    display_df["Historical Average"] = (
        display_df["Historical Average"].round(2)
    )

    display_df["Predicted Score"] = (
        display_df["Predicted Score"].round(2)
    )

    st.dataframe(
        display_df,
        use_container_width=True,
        hide_index=True
    )


    # --------------------------------------------------------
    # Performance chart
    # --------------------------------------------------------

    st.subheader("📈 Performance Trend")

    chart_data = results_df.set_index(
        "Subject"
    )[
        [
            "Period 1",
            "Period 2",
            "Period 3",
            "Period 4",
            "Predicted Score"
        ]
    ]

    st.line_chart(chart_data)


    # --------------------------------------------------------
    # Recommendations
    # --------------------------------------------------------

    st.subheader("💡 Recommendations")

    for _, row in results_df.iterrows():

        st.markdown(
            f"**{row['Subject']} — {row['Category']} / "
            f"{row['Trend']}**"
        )

        st.write(
            row["Recommendation"]
        )


    # --------------------------------------------------------
    # Model information
    # --------------------------------------------------------

    st.divider()

    st.subheader("🤖 Machine Learning Model")

    st.write(
        "Predictions are generated using a Random Forest "
        "Regression model trained on historical OULAD "
        "assessment data."
    )

    st.caption(
        "The deployed model uses four previous performance "
        "scores, their average, and the performance trend."
    )

Overwriting /content/app.py


In [174]:
!pkill -f "streamlit run" || true

^C


In [175]:
!streamlit run /content/app.py > /content/streamlit.log 2>&1 &

In [176]:

from pyngrok import ngrok

ngrok.set_auth_token("")

public_url = ngrok.connect(8501)

print("Open this URL:")
print(public_url)

Open this URL:
NgrokTunnel: "https://freeness-outbid-balcony.ngrok-free.dev" -> "http://localhost:8501"


In [177]:
!pkill -f "streamlit run" || true

!streamlit run /content/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    > /content/streamlit.log 2>&1 &

^C


In [178]:
!cat /content/streamlit.log

In [179]:
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [180]:
from google.colab import drive

drive.flush_and_unmount()

In [181]:
drive.mount('/content/drive')

Mounted at /content/drive


In [182]:
import os

PROJECT_DIR = "/content/drive/MyDrive/StudentProgressTracker"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/notebooks", exist_ok=True)

print("Project folder created:")
print(PROJECT_DIR)

Project folder created:
/content/drive/MyDrive/StudentProgressTracker


In [185]:
import shutil

shutil.copy(
    "/content/app.py",
    f"{PROJECT_DIR}/app.py"
)

shutil.copy(
    "/content/student_performance_model.pkl",
    f"{PROJECT_DIR}/models/student_performance_model.pkl"
)

shutil.copy(
    "/content/student_performance_features.pkl",
    f"{PROJECT_DIR}/models/student_performance_features.pkl"
)

print("Project files copied successfully!")

Project files copied successfully!
